In [11]:
import pandas as pd

df = pd.read_csv(r"C:\Users\ASUS\Downloads\archive (6)\Customer_support_data.csv")

df.head()

,Unique id,channel_name,category,Sub-category,Customer Remarks,Order_id,order_date_time,Issue_reported at,issue_responded,Survey_response_Date,Customer_City,Product_category,Item_price,connected_handling_time,Agent_name,Supervisor,Manager,Tenure Bucket,Agent Shift,CSAT Score
0,7e9ae164-6a8b-4521-a2d4-58f7c9fff13f,Outcall,Product Queries,Life Insurance,NaN,c27c9bb4-fa36-4140-9f1f-21009254ffdb,NaN,01-08-2023 11:13,01-08-2023 11:47,01-Aug-23,NaN,NaN,NaN,NaN,Richard Buchanan,Mason Gupta,Jennifer Nguyen,On Job Training,Morning,5
1,b07ec1b0-f376-43b6-86df-ec03da3b2e16,Outcall,Product Queries,Product Specific Information,NaN,d406b0c7-ce17-4654-b9de-f08d421254bd,NaN,01-08-2023 12:52,01-08-2023 12:54,01-Aug-23,NaN,NaN,NaN,NaN,Vicki Collins,Dylan Kim,Michael Lee,>90,Morning,5
2,200814dd-27c7-4149-ba2b-bd3af3092880,Inbound,Order Related,Installation/demo,NaN,c273368d-b961-44cb-beaf-62d6fd6c00d5,NaN,01-08-2023 20:16,01-08-2023 20:38,01-Aug-23,NaN,NaN,NaN,NaN,Duane Norman,Jackson Park,William Kim,On Job Training,Evening,5
3,eb0d3e53-c1ca-42d3-8486-e42c8d622135,Inbound,Returns,Reverse Pickup Enquiry,NaN,5aed0059-55a4-4ec6-bb54-97942092020a,NaN,01-08-2023 20:56,01-08-2023 21:16,01-Aug-23,NaN,NaN,NaN,NaN,Patrick Flores,Olivia Wang,John Smith,>90,Evening,5
4,ba903143-1e54-406c-b969-46c52f92e5df,Inbound,Cancellation,Not Needed,NaN,e8bed5a9-6933-4aff-9dc6-ccefd7dcde59,NaN,01-08-2023 10:30,01-08-2023 10:32,01-Aug-23,NaN,NaN,NaN,NaN,Christopher Sanchez,Austin Johnson,Michael Lee,0-30,Morning,5


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85907 entries, 0 to 85906
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unique id                85907 non-null  object 
 1   channel_name             85907 non-null  object 
 2   category                 85907 non-null  object 
 3   Sub-category             85907 non-null  object 
 4   Customer Remarks         28742 non-null  object 
 5   Order_id                 67675 non-null  object 
 6   order_date_time          17214 non-null  object 
 7   Issue_reported at        85907 non-null  object 
 8   issue_responded          85907 non-null  object 
 9   Survey_response_Date     85907 non-null  object 
 10  Customer_City            17079 non-null  object 
 11  Product_category         17196 non-null  object 
 12  Item_price               17206 non-null  float64
 13  connected_handling_time  242 non-null    float64
 14  Agent_name            

In [14]:
df['Issue_reported at'] = pd.to_datetime(df['Issue_reported at'])
df['issue_responded'] = pd.to_datetime(df['issue_responded'])

ValueError: time data "31-07-2023 23:58" doesn't match format "%m-%d-%Y %H:%M", at position 50. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
df['Issue_reported at'] = pd.to_datetime(df['Issue_reported at'], dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], dayfirst=True)

In [15]:
df[['Issue_reported at','issue_responded']].isna().sum()

Issue_reported at    0
issue_responded      0
dtype: int64

In [16]:
df = df.sort_values(by=['Order_id', 'Issue_reported at'])

In [17]:
df['repeat_contact_flag'] = df.duplicated(subset=['Order_id'], keep='first').astype(int)

In [18]:
df['repeat_contact_flag'].value_counts()

repeat_contact_flag
0    67676
1    18231
Name: count, dtype: int64

In [19]:
df['resolution_time_hours'] = (
    (df['issue_responded'] - df['Issue_reported at'])
    .dt.total_seconds() / 3600
)

TypeError: unsupported operand type(s) for -: 'str' and 'str'

In [20]:
df[['Issue_reported at','issue_responded']].dtypes


Issue_reported at    object
issue_responded      object
dtype: object

In [21]:
df['Issue_reported at'] = pd.to_datetime(
    df['Issue_reported at'], dayfirst=True, errors='coerce'
)

df['issue_responded'] = pd.to_datetime(
    df['issue_responded'], dayfirst=True, errors='coerce'
)

In [22]:
df[['Issue_reported at','issue_responded']].dtypes

Issue_reported at    datetime64[ns]
issue_responded      datetime64[ns]
dtype: object

In [23]:
df['resolution_time_hours'] = (
    (df['issue_responded'] - df['Issue_reported at'])
    .dt.total_seconds() / 3600
)

In [24]:
df['resolution_time_hours'].describe()

count    85907.000000
mean         2.281443
std          9.875476
min        -23.950000
25%          0.033333
50%          0.083333
75%          0.583333
max         95.966667
Name: resolution_time_hours, dtype: float64

In [25]:
df = df[df['resolution_time_hours'] >= 0]

In [26]:
df['resolution_time_hours'].describe()

count    82779.000000
mean         2.934399
std          9.419941
min          0.000000
25%          0.033333
50%          0.100000
75%          0.650000
max         95.966667
Name: resolution_time_hours, dtype: float64

In [27]:
df.groupby('repeat_contact_flag')['resolution_time_hours'].mean()

repeat_contact_flag
0    3.308196
1    1.610948
Name: resolution_time_hours, dtype: float64

In [28]:
df.groupby('category')['repeat_contact_flag'].mean().sort_values(ascending=False)

category
App/website           0.337349
Product Queries       0.296589
Offers & Cashback     0.271552
Shopzilla Related     0.269260
Payments related      0.247619
Returns               0.224999
Onboarding related    0.206897
Order Related         0.200413
Feedback              0.198730
Refund Related        0.194175
Cancellation          0.178605
Others                0.177083
Name: repeat_contact_flag, dtype: float64

In [29]:
df.groupby('Sub-category')['repeat_contact_flag'].mean().sort_values(ascending=False).head(10)

Sub-category
Policy Related                  0.500000
Issues with Shopzilla App       0.337838
Signup Issues                   0.334746
App/website Related             0.333333
Return cancellation             0.320144
e-Gift Voucher                  0.317073
Billing Related                 0.314815
Product related Issues          0.310734
Affiliate Offers                0.308571
Product Specific Information    0.299971
Name: repeat_contact_flag, dtype: float64

In [30]:
df.groupby('repeat_contact_flag')['resolution_time_hours'].describe()

,count,mean,std,min,25%,50%,75%,max
repeat_contact_flag,,,,,,,,
0,64548.0,3.308196,10.249454,0.0,0.033333,0.100000,0.766667,95.966667
1,18231.0,1.610948,5.359447,0.0,0.033333,0.083333,0.433333,95.483333


In [31]:
df.groupby('category').agg({
    'repeat_contact_flag': 'mean',
    'Order_id': 'count'
}).sort_values(by='Order_id', ascending=False)

,repeat_contact_flag,Order_id
category,,
Returns,0.224999,33053
Order Related,0.200413,17810
Refund Related,0.194175,3569
Product Queries,0.296589,2495
Shopzilla Related,0.269260,1934
Feedback,0.198730,1766
Cancellation,0.178605,1743
Payments related,0.247619,1659
Offers & Cashback,0.271552,338


In [32]:
df.groupby('repeat_contact_flag')['CSAT Score'].mean()

repeat_contact_flag
0    4.217590
1    4.304152
Name: CSAT Score, dtype: float64

In [33]:
df.groupby('repeat_contact_flag')['CSAT Score'].describe()

,count,mean,std,min,25%,50%,75%,max
repeat_contact_flag,,,,,,,,
0,64548.0,4.217590,1.399540,1.0,4.0,5.0,5.0,5.0
1,18231.0,4.304152,1.325485,1.0,4.0,5.0,5.0,5.0


In [34]:
df.groupby(['category','repeat_contact_flag'])['CSAT Score'].mean().unstack()

repeat_contact_flag,0,1
category,,
App/website,4.345455,4.500000
Cancellation,3.907057,4.345646
Feedback,4.145527,4.143836
Offers & Cashback,4.177515,4.309524
Onboarding related,4.043478,4.833333
Order Related,4.075182,4.138217
Others,3.493671,2.882353
Payments related,4.363472,4.289377
Product Queries,4.058918,3.942015


In [1]:
df.sample(1000).to_csv("sample_data_small.csv", index=False)

NameError: name 'df' is not defined

In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\ASUS\Downloads\archive (6)\Customer_support_data.csv")

In [3]:
df.sample(1000, random_state=42).to_csv("sample_data_small.csv", index=False)